# Q-ACCeSS-T: train RFR + optimize coefficients

Use in **Cursor** when VM CSV is huge (~5M rows).

1. VM collect → `derived/qaccess_training_samples.csv`
2. Copy CSV to Mac / set `CSV_PATH` below
3. Run all cells
4. scp `qaccess_t_best_coefficients.json` back to VM


In [ ]:
from pathlib import Path
REPO = Path.cwd()
if not (REPO / 'derived').exists() and (REPO.parent / 'derived').exists():
    REPO = REPO.parent
CSV_PATH = REPO / 'derived' / 'qaccess_training_samples.csv'
OUT_DIR = REPO / 'derived'
TRAIN_MAX_ROWS = 200_000
OPT_TAIL_ROWS = 50_000
OPT_MAX_SAMPLES = 500
N_ESTIMATORS = 80
MAX_DEPTH = 16
TEST_SIZE = 0.2
RANDOM_STATE = 42
print('REPO', REPO)
print('CSV exists', CSV_PATH.is_file())

In [ ]:
import importlib.util, subprocess, sys, json
from io import StringIO
for pkg, pip_name in [('pandas','pandas'),('numpy','numpy'),('sklearn','scikit-learn'),('joblib','joblib'),('matplotlib','matplotlib')]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable,'-m','pip','install',pip_name])
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
sys.path.insert(0, str(REPO / 'scripts' / 'analyze'))
from qaccess_math import candidate_triples, normalize_d, normalize_g, normalize_l, path_active, qaccess_gain_backoff, qaccess_utility
TARGET = 'next_bw_bps'
FEATURES = ['bw_bps','owd_ms','delay_gradient_ms','loss_rate','lost_bytes_delta','retrans_bytes_delta','cwnd_bytes','inflight_bytes','cwnd_room','alpha','beta','gamma','utility','gain','backoff']
print('imports OK')

In [ ]:
def load_csv_tail(path, n_rows):
    proc = subprocess.run(['tail','-n',str(n_rows+1),str(path)], capture_output=True, text=True, check=True)
    return pd.read_csv(StringIO(proc.stdout))

def prepare_frame(df):
    df = df.copy()
    df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce')
    df = df.dropna(subset=[TARGET])
    for col in FEATURES:
        if col not in df.columns: df[col] = 0.0
    return df

with open(CSV_PATH,'rb') as f:
    n_lines = sum(buf.count(b'\n') for buf in iter(lambda: f.read(1<<20), b''))
print(f'CSV lines (incl header): {n_lines:,}')

## 1. Train Random Forest

In [ ]:
df_train = prepare_frame(load_csv_tail(CSV_PATH, TRAIN_MAX_ROWS))
X = df_train[FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0.0)
y = df_train[TARGET].astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
model = RandomForestRegressor(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=RANDOM_STATE, n_jobs=-1)
print(f'Fitting on {len(X_tr):,} rows ...')
model.fit(X_tr, y_tr)
y_pred = model.predict(X_te)
mse = float(mean_squared_error(y_te, y_pred))
metrics = {'target':TARGET,'n_samples':len(df_train),'n_train':len(X_tr),'n_test':len(X_te),'train_max_rows':TRAIN_MAX_ROWS,'n_estimators':N_ESTIMATORS,'max_depth':MAX_DEPTH,'MSE':mse,'RMSE':float(np.sqrt(mse)),'MAE':float(mean_absolute_error(y_te,y_pred)),'R2':float(r2_score(y_te,y_pred)),'input_csv':str(CSV_PATH)}
print(json.dumps(metrics, indent=2))
imp = pd.DataFrame({'feature':FEATURES,'importance':model.feature_importances_}).sort_values('importance', ascending=False)
imp.head(10)

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, OUT_DIR/'qaccess_t_model.pkl')
(OUT_DIR/'qaccess_t_validation_metrics.json').write_text(json.dumps(metrics,indent=2)+'\n')
imp.to_csv(OUT_DIR/'qaccess_t_feature_importance.csv', index=False)
print('Saved model + metrics + importance under', OUT_DIR)
imp.head(10).plot.bar(x='feature', y='importance', figsize=(8,4), legend=False); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

## 2. Optimize alpha / beta / gamma

In [ ]:
def feature_matrix(samples, alpha, beta, gamma):
    rows = []
    for _, r in samples.iterrows():
        bw, owd = float(r.get('bw_bps',0) or 0), float(r.get('owd_ms',0) or 0)
        dgrad, loss = float(r.get('delay_gradient_ms',0) or 0), float(r.get('loss_rate',0) or 0)
        ng, nd, nl = normalize_g(bw), normalize_d(owd, dgrad), normalize_l(loss)
        u = qaccess_utility(ng, nd, nl, alpha, beta, gamma)
        gain, backoff = qaccess_gain_backoff(ng, nd, nl, alpha, beta, gamma)
        rows.append([bw,owd,dgrad,loss,float(r.get('lost_bytes_delta',0) or 0),float(r.get('retrans_bytes_delta',0) or 0),float(r.get('cwnd_bytes',0) or 0),float(r.get('inflight_bytes',0) or 0),float(r.get('cwnd_room',0) or 0),alpha,beta,gamma,u,gain,backoff])
    return np.asarray(rows, dtype=float)
df_opt = prepare_frame(load_csv_tail(CSV_PATH, OPT_TAIL_ROWS))
mask = df_opt.apply(lambda r: path_active(r.get('bw_bps',0), r.get('owd_ms',0), r.get('inflight_bytes',0)), axis=1)
samples = df_opt[mask].tail(OPT_MAX_SAMPLES)
best_a, best_b, best_g, best_pred = 0.70, 0.10, 0.10, -1.0
rows = []
for a,b,g in candidate_triples():
    pred = float(model.predict(feature_matrix(samples,a,b,g)).mean())
    rows.append({'alpha':a,'beta':b,'gamma':g,'pred':pred})
    if pred > best_pred: best_pred, best_a, best_b, best_g = pred, a, b, g
res_df = pd.DataFrame(rows).sort_values('pred', ascending=False)
display(res_df.head(10))
print(f'Best alpha={best_a} beta={best_b} gamma={best_g} pred={best_pred:.0f}')

In [ ]:
coeff = {'alpha':best_a,'beta':best_b,'gamma':best_g,'source':'qaccess_t_train_optimize.ipynb','metric':'predicted_next_bw_bps','predicted_next_bw_bps':best_pred,'n_samples':len(samples),'input_csv':str(CSV_PATH),'model':str(OUT_DIR/'qaccess_t_model.pkl')}
p = OUT_DIR/'qaccess_t_best_coefficients.json'
p.write_text(json.dumps(coeff, indent=2)+'\n')
print('Saved', p)
print(json.dumps(coeff, indent=2))